# UK Retail Intelligence using Apriori
## Data audit, basket construction and baseline association rules

Run from the repository root after installing `requirements.txt`. This notebook does not contain precomputed empirical findings.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.data import (download_dataset, extract_dataset, load_transactions,
                      audit_transactions, clean_transactions, build_basket_matrix)
from src.apriori import apriori, generate_rules
from src.validation import compare_itemsets


## 1. Acquire and audit the official UCI dataset

In [ ]:
archive = download_dataset()
workbook = extract_dataset(archive)
raw = load_transactions(workbook)
print('Raw shape:', raw.shape)
display(audit_transactions(raw))


## 2. Clean and construct UK invoice baskets

In [ ]:
clean, cleaning_counts = clean_transactions(raw, country='United Kingdom')
display(pd.Series(cleaning_counts, name='count').to_frame())
basket = build_basket_matrix(clean)
print('Basket matrix:', basket.shape)
print('Density:', basket.to_numpy().mean())


## 3. Chronological discovery and holdout split

The initial discovery cutoff is 31 August 2011. Threshold selection must use discovery data only.

In [ ]:
cutoff = pd.Timestamp('2011-08-31 23:59:59')
invoice_dates = clean.groupby('InvoiceNo')['InvoiceDate'].min()
discovery_ids = invoice_dates[invoice_dates <= cutoff].index
holdout_ids = invoice_dates[invoice_dates > cutoff].index
basket_discovery = basket.loc[basket.index.intersection(discovery_ids)]
basket_holdout = basket.loc[basket.index.intersection(holdout_ids)]
print('Discovery invoices:', len(basket_discovery))
print('Holdout invoices:', len(basket_holdout))


## 4. Mine frequent itemsets and generate rules

In [ ]:
transactions = [row.index[row.to_numpy(dtype=bool)].tolist()
                for _, row in basket_discovery.iterrows()]
itemsets = apriori(transactions, min_support=0.01, max_len=3)
rules = generate_rules(itemsets, min_confidence=0.20)
print('Frequent itemsets:', len(itemsets))
print('Rules:', len(rules))
display(itemsets.head(20))
display(rules.head(20))


## 5. Validate against mlxtend

In [ ]:
reference, own, comparison = compare_itemsets(
    basket_discovery, min_support=0.01, max_len=3
)
display(comparison['_merge'].value_counts(dropna=False))
print('Maximum support difference:', comparison['support_abs_difference'].max())
assert (comparison['_merge'] == 'both').all()
assert comparison['support_abs_difference'].dropna().max() < 1e-10


## 6. Research tasks before reporting conclusions

- Add a leakage-safe masked-item evaluation on the later holdout period.
- Compare against popularity and co-occurrence baselines.
- Report Precision@K, Recall@K, hit rate and rule coverage.
- Assess threshold sensitivity, temporal stability and rule redundancy.
- Treat association as co-occurrence, not causation.